# Simulating fleets of automated vehicles (AVs) making routing decisions: Small traffic network, DQN custom algorithm implementation

> In this notebook, on the `Two Route Yeild (TRY)` network, we simulate **10 human agents** for `2000 days`. After 100 days **10 of the human agents** mutate into automated vehicles (AVs) and use the [`IDQN`]((https://www.nature.com/articles/nature14236)) (Independent Q-Learning) algorithm. The AVs are `selfish` and their goal is to minimize their own travel time. 

---

#### Imported libraries

In [1]:
from tqdm import tqdm
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

#from extended_mutation import MyExtendedMutation
from routerl import TrafficEnvironment
from routerl import DQN
from routerl import Keychain as kc

os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

#### Hyperparameters setting

In [2]:
new_machines_after_mutation = 10
human_learning_episodes = 1
training_episodes = 2000
testing_episodes = 100

total_episodes = human_learning_episodes + training_episodes

env_params = {
    "agent_parameters" : {
        "new_machines_after_mutation": new_machines_after_mutation,

        "human_parameters" :
        {
            "model" : "general_model",

            "noise_weight_agent" : 0,
            "noise_weight_path" : 0.8,
            "noise_weight_day" : 0.2,

            "beta" : -1,
            "beta_k_i_variability" : 0.1,
            "epsilon_i_variability" : 0.1,
            "epsilon_k_i_variability" : 0.1,
            "epsilon_k_i_t_variability" : 0.1,

            "greedy" : 0.9,
            "gamma_c" : 0.0,
            "gamma_u" : 0.0,
            "remember" : 1,

            "alpha_zero" : 0.8,
            "alphas" : [0.2]  
        },
        "machine_parameters" :
        {
            "behavior" : "selfish",
            "observation_type" : "previous_agents_plus_start_time",
        }
    },
    "simulator_parameters" : {
        "network_name" : "two_route_yield",
        "sumo_type" : "sumo",
    },  
    "plotter_parameters" : {
        "phases" : [0, human_learning_episodes, int(training_episodes) + human_learning_episodes],
        "smooth_by" : 50,
        "phase_names" : [
            "Human learning", 
            "Mutation - Machine learning",
            "Testing phase"
        ],
        "plot_choices": "basic"
    },
    "path_generation_parameters":
    {
        "number_of_paths" : 2,
        "beta" : -1,
        "visualize_paths" : True
    }
}

#### Environment initialization

> In this example, the environment initially contains only human agents.

> If the paths are already created then create_paths=False, we don't have to create again.

In [3]:
env = TrafficEnvironment(seed=42, create_agents=False, create_paths=True, marginal_cost_calculation=False, **env_params)

In [4]:
print("Number of total agents is: ", len(env.all_agents), "\n")
print("Number of human agents is: ", len(env.human_agents), "\n")
print("Number of machine agents (autonomous vehicles) is: ", len(env.machine_agents), "\n")

Number of total agents is:  22 

Number of human agents is:  22 

Number of machine agents (autonomous vehicles) is:  0 



> Reset the environment and the connection with SUMO

In [5]:
env.start()

In [6]:
env.reset()

({}, {})

#### Human learning

In [7]:
pbar = tqdm(total=total_episodes, desc="Human learning")

for episode in range(human_learning_episodes):
    env.step()

Human learning:   0%|          | 0/2001 [00:00<?, ?it/s]

#### Mutation

> **Mutation**: a portion of human agents are converted into machine agents (autonomous vehicles). 

In [8]:
pre_mutation_agents = env.all_agents.copy()

env.mutation_odd_id_agents()

> Humans are fixed to action zero in this case.

In [9]:
for human in env.human_agents:
    human.default_action = 0

In [10]:
machines = env.machine_agents.copy()
mutated_humans = dict()
for machine in machines:
    for human in pre_mutation_agents:
        if human.id == machine.id:
            mutated_humans[str(machine.id)] = human
            break

> Each AV agent has an independent learning component.

In [11]:
free_flows = env.get_free_flow_times()
for h_id, human in mutated_humans.items():
    initial_knowledge = free_flows[(human.origin, human.destination)]
    initial_knowledge = [0, 0]
    mutated_humans[h_id].model = DQN(3, len(initial_knowledge))

In [12]:
mutated_humans

{'1': Human 1,
 '3': Human 3,
 '5': Human 5,
 '7': Human 7,
 '9': Human 9,
 '11': Human 11,
 '13': Human 13,
 '15': Human 15,
 '17': Human 17,
 '19': Human 19}

#### AV's training loop

In [13]:
pbar.set_description("AV learning")
os.makedirs("plots", exist_ok=True)
for episode in range(training_episodes):
    env.reset()
    for agent in env.agent_iter():
        observation, reward, termination, truncation, info = env.last()
        
        if termination or truncation:
            obs = [{kc.AGENT_ID : int(agent), kc.TRAVEL_TIME : -reward}]
            last_action = mutated_humans[agent].last_action
            last_observation = mutated_humans[agent].last_obs
            
            mutated_humans[agent].learn(last_action, obs)
            action = None
        else:
            action = mutated_humans[agent].act(observation)
            mutated_humans[agent].last_action = action

        env.step(action)
    """if episode % 50 == 0:
        env.plot_results()"""
    pbar.update()

AV learning: 100%|█████████▉| 2000/2001 [09:41<00:00,  4.06it/s]

#### AV's testing loop

In [14]:
pbar.set_description("Testing")
for episode in range(testing_episodes):
    env.reset()
    for agent in env.agent_iter():
        observation, reward, termination, truncation, info = env.last()
        if termination or truncation:
            action = None
        else:
            action = mutated_humans[agent].act(observation)
        env.step(action)
    pbar.update()

pbar.close()
#os.makedirs(plots_folder, exist_ok=True)


Testing: : 2100it [10:09,  3.45it/s]                            


>  Check `\plots` directory to find the plots created from this experiment.

In [15]:
env.plot_results()

> Interrupt the connection with `SUMO`.

In [16]:
env.stop_simulation()